# Week 75, Day 3: sqlite3

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w075d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE users (id INTEGER PRIMARY KEY, name TEXT)")
conn.executemany("INSERT INTO users (name) VALUES (?)", [("ada",), ("bob",)])

name = "nobody' OR '1'='1"
unsafe = conn.execute(f"SELECT count(*) FROM users WHERE name = '{name}'").fetchone()
safe = conn.execute("SELECT count(*) FROM users WHERE name = ?", (name,)).fetchone()
print(unsafe, safe)

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE notes (id INTEGER PRIMARY KEY, title TEXT NOT NULL, body TEXT)")
cur = conn.execute("INSERT INTO notes (title, body) VALUES (?, ?)", ("Shopping", "tea, jam"))
print("new id:", cur.lastrowid)
conn.executemany("INSERT INTO notes (title, body) VALUES (:t, :b)",
                 [{"t": "Ideas", "b": "a garden for bees"}, {"t": "O'Brien's party", "b": None}])
word = "bee"
print(conn.execute("SELECT id, title FROM notes WHERE body LIKE ?", (f"%{word}%",)).fetchall())
cur = conn.execute("DELETE FROM notes WHERE title LIKE ?", ("%party%",))
print("deleted:", cur.rowcount, "left:", conn.execute("SELECT count(*) FROM notes").fetchone()[0])

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE accounts (name TEXT PRIMARY KEY, balance INTEGER NOT NULL CHECK (balance >= 0))")
with conn:
    conn.executemany("INSERT INTO accounts VALUES (?, ?)", [("ada", 100), ("bob", 20)])

def transfer(conn, source, target, amount):
    with conn:                                   # one transaction: both updates or neither
        conn.execute("UPDATE accounts SET balance = balance + ? WHERE name = ?", (amount, target))
        conn.execute("UPDATE accounts SET balance = balance - ? WHERE name = ?", (amount, source))

transfer(conn, "ada", "bob", 30)
try:
    transfer(conn, "bob", "ada", 500)            # the second update breaks the CHECK, so the first is undone
except sqlite3.IntegrityError as e:
    print("refused:", e)
print(conn.execute("SELECT * FROM accounts ORDER BY name").fetchall())

In [ ]:
import sqlite3
from datetime import date

conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
conn.execute("PRAGMA foreign_keys = ON")
conn.executescript("""
CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT NOT NULL);
CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER NOT NULL REFERENCES customers(id), placed TEXT NOT NULL);
INSERT INTO customers VALUES (1, 'Ada');
""")
with conn:
    conn.execute("INSERT INTO orders (customer_id, placed) VALUES (?, ?)", (1, date(2025, 3, 29).isoformat()))
row = conn.execute("SELECT o.id, c.name, o.placed FROM orders o JOIN customers c ON c.id = o.customer_id").fetchone()
print(dict(row), date.fromisoformat(row["placed"]).strftime("%A"))
try:
    with conn:
        conn.execute("INSERT INTO orders (customer_id, placed) VALUES (99, '2025-04-01')")
except sqlite3.IntegrityError as e:
    print("refused:", e)

> **Key idea.** connect opens a database (a file or :memory:); execute returns a cursor to iterate or fetchone/fetchall; executemany repeats a statement, executescript runs a script (committing first); lastrowid and rowcount report changes; close connections. Always pass values as parameters (? with a tuple, :name with a dict); never format them into SQL, which allows injection; names can't be parameters, so pick them from a fixed list. Changes open a transaction automatically and need commit; with conn commits or rolls back (but doesn't close); DDL and PRAGMA need an explicit BEGIN to be transactional. row_factory = sqlite3.Row gives dict-like rows; convert dates and other types at the boundary (or with adapters and converters); store money as integers; turn on PRAGMA foreign_keys per connection.

### Exercise: A small notes store

Write four functions over a connection. `setup(conn)` creates a table `notes` with `id` (integer primary key), `title` (text, required) and `body` (text, may be NULL). `add(conn, title, body=None)` inserts a note, commits, and returns its new id. `search(conn, word)` returns `(id, title)` for notes whose title **or** body contains `word` (any case), by id, passing the pattern as a parameter. `delete(conn, note_id)` deletes a note, commits, and returns whether a row was deleted.

In [ ]:
import sqlite3

def setup(conn):
    ___

def add(conn, title, body=None):
    ___

def search(conn, word):
    ___

def delete(conn, note_id):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_notes():\n    \"insert, search and delete, safely\"\n    import sqlite3\n    conn = sqlite3.connect(\":memory:\")\n    setup(conn)\n    a = add(conn, \"Shopping\", \"Tea and JAM\")\n    b = add(conn, \"Jam recipe\")\n    c = add(conn, \"O'Brien's party\", \"bring jam\")\n    assert (a, b, c) == (1, 2, 3)\n    assert search(conn, \"jam\") == [(1, \"Shopping\"), (2, \"Jam recipe\"), (3, \"O'Brien's party\")]\n    assert search(conn, \"'\") == [(3, \"O'Brien's party\")] and search(conn, \"x' OR '1'='1\") == []\n    assert delete(conn, 2) is True and delete(conn, 2) is False\n    assert not conn.in_transaction, \"commit your changes\"\n    assert \"{\" not in source.split(\"def search\")[1].split(\"def delete\")[0] or \"?\" in source")

### Exercise: Loading many rows

Write `load_products(conn, csv_text)`: create a table `products (sku TEXT PRIMARY KEY, name TEXT NOT NULL, pence INTEGER NOT NULL)` if it doesn't exist, read `csv_text` (with a header row `sku,name,price`, where the price is in pounds such as `3.50`) with the `csv` module, and insert every row with **one** `executemany`, storing the price as a whole number of pence (round it, don't truncate). Commit, and return the number of rows now in the table. A duplicate SKU should roll the whole load back and raise `sqlite3.IntegrityError`.

In [ ]:
import csv, io, sqlite3

def load_products(conn, csv_text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_bulk():\n    \"executemany and all-or-nothing loading\"\n    import sqlite3\n    conn = sqlite3.connect(\":memory:\")\n    assert load_products(conn, \"sku,name,price\\nT1,Tea,3.50\\nC1,Coffee,6.00\\nB1,Biscuits,1.15\\n\") == 3\n    assert conn.execute(\"SELECT pence FROM products WHERE sku = 'B1'\").fetchone() == (115,)\n    try:\n        load_products(conn, \"sku,name,price\\nJ1,Jam,4.25\\nT1,Tea again,3.00\\n\")\n        assert False\n    except sqlite3.IntegrityError:\n        pass\n    assert conn.execute(\"SELECT count(*) FROM products\").fetchone() == (3,), \"the failed load must be rolled back\"\n    assert \"executemany\" in source")

### Exercise: Rows as dictionaries

Write `dict_factory(cursor, row)`, a row factory that returns a plain `dict` of column name to value, and `query(conn, sql, params=())`: run a query with `dict_factory` as the row factory **for that query only** (set it on a new cursor from `conn.cursor()`, leaving `conn.row_factory` untouched), returning a list of dicts. Then `one(conn, sql, params=())`: the single row as a dict, raising `LookupError` if there are none and `ValueError` if there are several.

In [ ]:
import sqlite3

def dict_factory(cursor, row):
    ___

def query(conn, sql, params=()):
    ___

def one(conn, sql, params=()):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_dicts():\n    \"named columns\"\n    import sqlite3\n    conn = sqlite3.connect(\":memory:\")\n    conn.executescript(\"CREATE TABLE t (id INTEGER, name TEXT); INSERT INTO t VALUES (1, 'a'), (2, 'b');\")\n    assert query(conn, \"SELECT id, name AS label FROM t ORDER BY id\") == [{\"id\": 1, \"label\": \"a\"}, {\"id\": 2, \"label\": \"b\"}]\n    assert one(conn, \"SELECT name FROM t WHERE id = ?\", (2,)) == {\"name\": \"b\"}\n    assert conn.execute(\"SELECT id FROM t WHERE id = 1\").fetchone() == (1,), \"conn.row_factory should be unchanged\"\n    for sql, exc in [(\"SELECT * FROM t WHERE id = 9\", LookupError), (\"SELECT * FROM t\", ValueError)]:\n        try:\n            one(conn, sql)\n            assert False\n        except exc:\n            pass")

### Exercise: All or nothing

Write `setup_bank(conn)` (a table `accounts (name TEXT PRIMARY KEY, pence INTEGER NOT NULL CHECK (pence >= 0))` and a table `transfers (id INTEGER PRIMARY KEY, source TEXT, target TEXT, pence INTEGER)`), and `transfer(conn, source, target, pence)`: in **one transaction**, take the money from `source`, give it to `target`, and record the transfer; return the transfer's id. If an account doesn't exist (an update changes no rows), raise `KeyError` naming it; if the source can't cover it, let the `IntegrityError` from the check propagate. Either way, **nothing** must change. `pence` must be a positive integer (`ValueError` otherwise).

In [ ]:
import sqlite3

def setup_bank(conn):
    ___

def transfer(conn, source, target, pence):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_transfer():\n    \"atomic transfers\"\n    import sqlite3\n    conn = sqlite3.connect(\":memory:\")\n    setup_bank(conn)\n    with conn:\n        conn.executemany(\"INSERT INTO accounts VALUES (?, ?)\", [(\"ada\", 10000), (\"bob\", 500)])\n    assert transfer(conn, \"ada\", \"bob\", 2500) == 1\n    def state():\n        return conn.execute(\"SELECT * FROM accounts ORDER BY name\").fetchall(), conn.execute(\"SELECT count(*) FROM transfers\").fetchone()[0]\n    assert state() == ([(\"ada\", 7500), (\"bob\", 3000)], 1)\n    for args, exc in [((\"bob\", \"ada\", 99999), sqlite3.IntegrityError), ((\"ada\", \"cy\", 10), KeyError), ((\"cy\", \"ada\", 10), KeyError), ((\"ada\", \"bob\", 0), ValueError), ((\"ada\", \"bob\", 1.5), ValueError)]:\n        try:\n            transfer(conn, *args)\n            assert False, args\n        except exc:\n            pass\n        assert state() == ([(\"ada\", 7500), (\"bob\", 3000)], 1), f\"something changed after {args}\"")

### Exercise: Dates in and out

Write `open_diary()`: return a new in-memory connection opened with `detect_types=sqlite3.PARSE_DECLTYPES`, after registering an adapter that stores `datetime.date` values as ISO text and a converter for the declared type `DATE` that turns them back into `date` objects; create a table `entries (day DATE NOT NULL, text TEXT NOT NULL)`. Then `between(conn, start, end)`: the `(day, text)` rows with `start <= day <= end` (passing `date` objects as parameters), by day. Finally register a SQL function `weekday(day)` on the connection (in `open_diary`) returning the day's English name, such as `'Monday'`, from its ISO text.

In [ ]:
import sqlite3
from datetime import date

def open_diary():
    ___

def between(conn, start, end):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_dates():\n    \"adapters, converters and functions\"\n    from datetime import date\n    conn = open_diary()\n    with conn:\n        conn.executemany(\"INSERT INTO entries VALUES (?, ?)\", [(date(2025, 3, 29), \"rain\"), (date(2025, 4, 2), \"sun\"), (date(2025, 5, 1), \"wind\")])\n    got = between(conn, date(2025, 3, 1), date(2025, 4, 30))\n    assert got == [(date(2025, 3, 29), \"rain\"), (date(2025, 4, 2), \"sun\")] and isinstance(got[0][0], date)\n    assert conn.execute(\"SELECT weekday(day) FROM entries ORDER BY day\").fetchall() == [(\"Saturday\",), (\"Wednesday\",), (\"Thursday\",)]")

### Exercise: Keys that are checked

Write `open_shop()`: an in-memory connection with foreign keys **turned on**, and tables `customers (id INTEGER PRIMARY KEY, name TEXT NOT NULL)`, `orders (id INTEGER PRIMARY KEY, customer_id INTEGER NOT NULL REFERENCES customers(id) ON DELETE CASCADE)` and `reviews (id INTEGER PRIMARY KEY, customer_id INTEGER REFERENCES customers(id) ON DELETE SET NULL, text TEXT)`. Then `remove_customer(conn, customer_id)`: delete the customer (their orders go with them; their reviews stay, with no customer), commit, and return `(orders_left, reviews_left)` in the whole database.

In [ ]:
import sqlite3

def open_shop():
    ___

def remove_customer(conn, customer_id):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_foreign():\n    \"enforced references\"\n    import sqlite3\n    conn = open_shop()\n    with conn:\n        conn.executescript(\"INSERT INTO customers VALUES (1, 'Ada'), (2, 'Bob'); INSERT INTO orders VALUES (1, 1), (2, 1), (3, 2); INSERT INTO reviews VALUES (1, 1, 'great'), (2, 2, 'fine');\")\n    try:\n        with conn:\n            conn.execute(\"INSERT INTO orders VALUES (4, 99)\")\n        assert False, \"an order for a missing customer should be refused\"\n    except sqlite3.IntegrityError:\n        pass\n    assert remove_customer(conn, 1) == (1, 2)\n    assert conn.execute(\"SELECT customer_id FROM reviews ORDER BY id\").fetchall() == [(None,), (2,)]")

### Bug hunt: A search anyone can break

The search pastes the user's text into the SQL, so the input `x' OR '1'='1` matches **every** row and this prints `3`. Pass the value as a parameter, so this prints `0`.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE users (name TEXT)")
conn.executemany("INSERT INTO users VALUES (?)", [("ada",), ("bob",), ("cy",)])

def find(name):
    return conn.execute(f"SELECT * FROM users WHERE name = '{name}'").fetchall()

print(len(find("x' OR '1'='1")))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a parameter, not formatting\"\n    expect_output(\"0\")\n    assert \"f\\\"SELECT\" not in source and \"f'SELECT\" not in source")

### Bug hunt: Saved, until it wasn't

The program inserts a row and closes the connection without committing, so the insert is rolled back and a new connection to the same file sees nothing: this prints `0`. Commit the change (with `with conn:` around the insert), so it prints `1`.

In [ ]:
import os, sqlite3

if os.path.exists("diary.db"):
    os.remove("diary.db")
conn = sqlite3.connect("diary.db")
conn.execute("CREATE TABLE entries (text TEXT)")
conn.execute("INSERT INTO entries VALUES ('a good day')")
conn.close()

again = sqlite3.connect("diary.db")
print(again.execute("SELECT count(*) FROM entries").fetchone()[0])
again.close()

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"commit before closing\"\n    expect_output(\"1\")")

### Bug hunt: One parameter that wasn't a sequence

`execute` expects a **sequence** of parameters, and `"ada"` is a sequence of three characters, so SQLite complains that one placeholder was given three values. Pass a one-item tuple, so this prints `[(1, 'ada')]`.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.executescript("CREATE TABLE users (id INTEGER, name TEXT); INSERT INTO users VALUES (1, 'ada'), (2, 'bob');")
print(conn.execute("SELECT * FROM users WHERE name = ?", "ada").fetchall())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a one-item tuple\"\n    expect_output(\"[(1, 'ada')]\")")

### Puzzle: A safe, atomic update

Arrange the lines of a function that marks an order as shipped and records when, as one transaction, with parameters. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # True or False: "with conn:" closes the connection at the end of the block
q = ___   # what cur.rowcount is after an UPDATE that matched no rows
r = ___   # the placeholder style that takes a dict: "?" or ":name"
s = ___   # True or False: a table name can be passed as a ? parameter
t = ___   # the PRAGMA that makes SQLite check REFERENCES constraints

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p is False and q == 0 and r == \":name\", \"p, q and r\"\n    assert s is False and t in (\"foreign_keys\", \"PRAGMA foreign_keys = ON\", \"foreign_keys = ON\"), \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: schema migrations

Applications change their schema over time, so they keep a numbered list of **migrations** and record which have been applied. Write `migrate(conn, migrations)`: `migrations` is a list of lists of SQL statements, where migration number `n` (from 1) is `migrations[n - 1]`. The database's current version is `PRAGMA user_version` (0 for a new database). Apply each pending migration in order, **each in its own transaction**: start it with an explicit `BEGIN`, run its statements one by one, set `PRAGMA user_version` to its number, and commit. If a statement fails, roll that migration back (its earlier statements and the version change are undone), stop, and raise `RuntimeError(f"migration {n} failed: {error}")`; migrations before it stay applied. Return the list of migration numbers applied by this call. Also `version(conn)`.

In [ ]:
import sqlite3

def version(conn):
    ___

def migrate(conn, migrations):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_migrations():\n    \"versioned, atomic schema changes\"\n    import sqlite3\n    conn = sqlite3.connect(\":memory:\")\n    steps = [\n        [\"CREATE TABLE notes (id INTEGER PRIMARY KEY, title TEXT NOT NULL)\"],\n        [\"ALTER TABLE notes ADD COLUMN body TEXT\", \"CREATE INDEX notes_title ON notes (title)\"],\n    ]\n    assert version(conn) == 0 and migrate(conn, steps) == [1, 2] and version(conn) == 2\n    assert migrate(conn, steps) == []\n    conn.execute(\"INSERT INTO notes (title, body) VALUES ('a', 'b')\")\n    conn.commit()\n    broken = steps + [[\"CREATE TABLE tags (name TEXT)\", \"INSERT INTO nowhere VALUES (1)\"], [\"CREATE TABLE later (x)\"]]\n    try:\n        migrate(conn, broken)\n        assert False\n    except RuntimeError as e:\n        assert str(e).startswith(\"migration 3 failed: \")\n    tables = {r[0] for r in conn.execute(\"SELECT name FROM sqlite_master WHERE type = 'table'\")}\n    assert version(conn) == 2 and \"tags\" not in tables and \"later\" not in tables\n    fixed = steps + [[\"CREATE TABLE tags (name TEXT)\"], [\"CREATE TABLE later (x)\"]]\n    assert migrate(conn, fixed) == [3, 4] and version(conn) == 4\n    assert conn.execute(\"SELECT title, body FROM notes\").fetchall() == [(\"a\", \"b\")]")

## Warm-up

The same "name" found **every** user when it was pasted into the SQL text, because the quote inside it ended the string and `OR '1'='1'` became part of the query. Passed as a **parameter**, it's just a strange name, and matches nobody. That difference is **SQL injection**, one of the most common security holes there is (Week 79), and it's why today's first rule is: never build SQL from values with string formatting.

## Connections and cursors

```python
import sqlite3

conn = sqlite3.connect("shop.db")        # a file, created if needed; ":memory:" for a temporary one
cur = conn.execute("SELECT id, name FROM products WHERE price < ?", (5,))
for row in cur:                          # rows are tuples by default
    print(row)
conn.close()
```

- **`sqlite3.connect(path)`** opens (or creates) a database. A **connection** is your session with it: it holds the current transaction, and must be closed when you're done (`conn.close()`, or `contextlib.closing(conn)` in a `with`).
- **`conn.execute(sql, params)`** runs one statement and returns a **cursor**: an object for reading the results. Iterate over it, or call `fetchone()` (one row, or `None`), `fetchmany(n)` or `fetchall()` (a list). `cur.description` names the columns.
- **`executemany(sql, sequence_of_params)`** runs one statement for many sets of values: the fast way to insert many rows. **`executescript(text)`** runs several statements at once (for setting up a schema), but it **commits** first and doesn't take parameters.
- After an `INSERT`, **`cur.lastrowid`** is the new row's id; after `UPDATE` or `DELETE`, **`cur.rowcount`** says how many rows changed.

## Parameters, never formatting

Put values into SQL with **placeholders**, and pass the values separately:

```python
conn.execute("SELECT * FROM products WHERE category = ? AND price < ?", ("food", 4))
conn.execute("SELECT * FROM products WHERE category = :cat", {"cat": "food"})
```

- `?` placeholders take a **sequence** (a tuple or list), in order; `:name` placeholders take a dict. A single value still needs a tuple: `(5,)`, not `5`.
- The driver sends the values **separately from the SQL**, so they can never change the query's structure: quotes, semicolons and `OR 1=1` in a value are just characters.
- Only **values** can be parameters, not table or column names or SQL keywords. When those must vary, choose them from a fixed list in your code (as in Day 1's query builder), never from user input directly.
- Parameters also handle types and escaping for you (`None` becomes `NULL`), and let the database reuse the compiled query.

`LIKE` patterns are values too: build the pattern in Python and pass it: `("%" + word + "%",)`.

## Transactions

A **transaction** is a group of changes that happen **all together or not at all**: moving money takes it out of one account and puts it into another, and a crash half-way must never leave just one of those done. Changes inside a transaction aren't seen by other connections, and aren't saved, until you **commit**; a **rollback** undoes them all.

How Python's `sqlite3` handles this by default:

- Before an `INSERT`, `UPDATE`, `DELETE` or `REPLACE`, it **opens a transaction automatically** if none is open. The changes stay pending until **`conn.commit()`**, or are discarded by **`conn.rollback()`** (or by closing the connection without committing).
- **`with conn:`** commits when the block ends normally and **rolls back** if it raises. It does **not** close the connection.
- Other statements, including `CREATE TABLE` and `PRAGMA`, don't open a transaction by themselves, so they take effect at once. To make schema changes all-or-nothing, start the transaction explicitly with `conn.execute("BEGIN")`, and then `commit()` or `rollback()`.
- Because a connection holds an open transaction until you commit, forgetting `commit()` is the classic bug: your program seems to work, and then the data's gone the next time it runs.

## Rows, types and foreign keys

**Rows as dictionaries.** Set **`conn.row_factory`** to change what rows are: `sqlite3.Row` gives rows that work like tuples **and** like dictionaries (`row["name"]`, `row.keys()`); or write your own factory, `lambda cursor, row: dict(zip([d[0] for d in cursor.description], row))`, for plain dicts.

**Types.** SQLite stores `NULL`, integers, floats, text and bytes, and gives them back as `None`, `int`, `float`, `str` and `bytes`. For other Python types, convert at the boundary:

- store dates as ISO text (`d.isoformat()`) and parse them back (`date.fromisoformat(text)`), explicitly, or
- register an **adapter** (`sqlite3.register_adapter(date, date.isoformat)`, from Python to SQLite) and a **converter** (`sqlite3.register_converter("DATE", lambda b: date.fromisoformat(b.decode()))`, from SQLite to Python, chosen by the column's declared type when you connect with `detect_types=sqlite3.PARSE_DECLTYPES`).
- Money is safest as an integer number of pence (or cents), never as a float.

**Foreign keys.** SQLite checks `REFERENCES` constraints only after **`PRAGMA foreign_keys = ON`**, which must be set on **each connection** (most other databases always check). Then inserting an order for a customer who doesn't exist, or deleting a customer who still has orders, raises `IntegrityError`, unless the key says what should happen: `ON DELETE CASCADE` (delete their orders too) or `ON DELETE SET NULL`.

**Your own SQL functions.** `conn.create_function("slug", 1, slugify)` makes a Python function callable from SQL as `slug(title)`.

## Practice

## On your own computer

The same code works with a database server, through a different driver with the same interface (the **DB-API**, PEP 249): `psycopg` for PostgreSQL uses `%s` placeholders instead of `?`, but `connect`, `execute`, `fetchall`, `commit` and `with conn` all carry over:

```bash
pip install "psycopg[binary]"
```

```python
import psycopg

with psycopg.connect("dbname=shop user=me") as conn:
    rows = conn.execute("SELECT name FROM products WHERE price < %s", (5,)).fetchall()
```

The connection details belong in configuration or environment variables, never in code (Week 79).

## Quiz

## Daily challenge

## Before you go

Find a program from an earlier week that saved data to a CSV or JSON file, and sketch the tables it would need instead. Tomorrow: **ORMs**, which map those tables to Python classes, and the trade-offs that come with letting a library write your SQL.